In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg

# Forzamos el cierre de cualquier sesión anterior para limpiar puertos
try:
    spark.stop()
except:
    pass

# 1. Iniciar una SparkSession totalmente nueva en el puerto 4040
spark = SparkSession.builder \
    .appName("Benchmark-Lambda-Batch") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

print("=== INICIANDO BENCHMARK: LAMBDA (BATCH LAYER) ===")
start_time = time.time()

# 2. Generar Dataset sintético para el Benchmark
df_batch = spark.range(0, 1000000) \
    .withColumn("sensor_id", (col("id") % 100).cast("string")) \
    .withColumn("temperatura", (col("id") % 50) + 15.5) \
    .withColumn("humedad", (col("id") % 40) + 30.0)

# 3. Procesamiento Batch: Agregaciones analíticas masivas
df_metrics = df_batch.groupBy("sensor_id") \
    .agg(
        count("id").alias("total_lecturas"),
        avg("temperatura").alias("temp_promedio"),
        avg("humedad").alias("humedad_promedio")
    )

# Acción de Spark
total_filas = df_metrics.count()
end_time = time.time()

execution_time = end_time - start_time
print(f"Registros procesados en Batch: {total_filas}")
print(f"⏱️ Tiempo total de ejecución Batch: {execution_time:.4f} segundos")

print("\n🔍 Manteniendo Spark activo por 60 segundos para que puedas ver la Spark UI...")
time.sleep(60)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 03:50:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== INICIANDO BENCHMARK: LAMBDA (BATCH LAYER) ===


Registros procesados en Batch: 100
⏱️ Tiempo total de ejecución Batch: 4.7369 segundos

🔍 Manteniendo Spark activo por 60 segundos para que puedas ver la Spark UI...
